In [5]:
# Install required packages
!pip install -q pandas numpy scikit-learn xgboost lightgbm

'c:\Users\Vedant Singh\AppData\Roaming\Python\Python311\Scripts\pip.exe' was blocked by your organization's Device Guard policy.
Contact your support person for more info.


In [7]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# Load data
# Replace 'data.csv' with your dataset path
import os
file_path = os.path.join(os.getcwd(), 'data.csv')
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    # Create a dummy DataFrame if file not found
    df = pd.DataFrame({'feature1': np.random.rand(100),
                       'feature2': np.random.rand(100),
                       'price': np.random.rand(100)*100})
print(f"Data shape: {df.shape}")

In [ ]:
# Preprocess data
# Simple example: drop rows with missing target and fill other missing values
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate target
if 'price' in df.columns:
    y = df['price']
    X = df.drop(columns=['price'])
else:
    raise KeyError("Target column 'price' not found in dataframe.")

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

In [ ]:
# Train and evaluate models
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build pipelines with preprocessing
rf_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                         ('model', RandomForestRegressor(n_estimators=200, random_state=42))])
xgb_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', xgb.XGBRegressor(objective='reg:squarederror', n_estimators=200, learning_rate=0.05, random_state=42))])
lgb_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42))])

# Train models
rf_pipe.fit(X_train, y_train)
xgb_pipe.fit(X_train, y_train)
lgb_pipe.fit(X_train, y_train)

# Predict
rf_pred = rf_pipe.predict(X_test)
xgb_pred = xgb_pipe.predict(X_test)
lgb_pred = lgb_pipe.predict(X_test)

# Evaluate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} - MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

print("Model evaluation: ")
eval_metrics(y_test, rf_pred, "Random Forest")
eval_metrics(y_test, xgb_pred, "XGBoost")
eval_metrics(y_test, lgb_pred, "LightGBM")